## Building Linear regression
<p>
Building a linear regression using linear algebra
</p>

**Formula**
$$\hat{\beta} = (X^T X)^{-1} X^T y$$



In [3]:
import numpy as np
import pandas as pd

class AdvancedLinearRegression:
    def __init__(self, fit_intercept=True):
        self.fit_intercept = fit_intercept
        self.coefficients = None
        self.intercept = None
        self.feature_names = None
        self.selected_features = None

    def _add_intercept(self, X):
        ones = np.ones((X.shape[0], 1))
        return np.hstack((ones, X))

    def fit(self, X, y, feature_names=None):
        """Fits the linear regression model using the Normal Equation."""
        if isinstance(X, pd.DataFrame):
            self.feature_names = X.columns.tolist()
            X_mat = X.to_numpy()
        else:
            self.feature_names = feature_names if feature_names else [f"X{i}" for i in range(X.shape[1])]
            X_mat = np.asarray(X)
            
        y_mat = np.asarray(y).reshape(-1, 1)

        if self.fit_intercept:
            X_mat = self._add_intercept(X_mat)

        # Normal Equation: Beta = (X^T * X)^(-1) * X^T * y
        XtX = np.dot(X_mat.T, X_mat)
        XtY = np.dot(X_mat.T, y_mat)
        
        try:
            beta = np.dot(np.linalg.inv(XtX), XtY)
        except np.linalg.LinAlgError:
            # Handle multicollinearity using pseudo-inverse if XtX is singular
            beta = np.dot(np.linalg.pinv(XtX), XtY)

        if self.fit_intercept:
            self.intercept = float(beta[0])
            self.coefficients = beta[1:].flatten()
        else:
            self.intercept = 0.0
            self.coefficients = beta.flatten()
            
        return self

    def predict(self, X):
        """Predicts using the linear model."""
        X_mat = np.asarray(X)
        predictions = np.dot(X_mat, self.coefficients) + self.intercept
        return predictions.flatten()

    def get_coefficients(self):
        """Returns coefficients mapped to feature names."""
        coef_dict = {name: coef for name, coef in zip(self.feature_names, self.coefficients)}
        if self.fit_intercept:
            coef_dict['Intercept'] = self.intercept
        return coef_dict

    # --- ADVANCED CAPABILITIES ---

    def simulate_sensitivity(self, X, feature_name, perturbation_percentage=5.0):
        """
        Simulates how sensitive predictions are to changes in a specific feature.
        Perturbs the feature by ± percentage and calculates average target shift.
        """
        if feature_name_prejudices not in self.feature_names:
            raise ValueError(f"Feature '{feature_name}' not found in trained model.")
            
        X_base = pd.DataFrame(X, columns=self.feature_names).copy()
        idx = self.feature_names.index(feature_name)
        
        # Baseline predictions
        base_preds = self.predict(X_base)
        
        # Perturb up and down
        factor = perturbation_percentage / 100.0
        X_up = X_base.copy(); X_up.iloc[:, idx] *= (1 + factor)
        X_down = X_base.copy(); X_down.iloc[:, idx] *= (1 - factor)
        
        preds_up = self.predict(X_up)
        preds_down = self.predict(X_down)
        
        mean_pct_change_up = np.mean((preds_up - base_preds) / (base_preds + 1e-9)) * 100
        mean_pct_change_down = np.mean((preds_down - base_preds) / (base_preds + 1e-9)) * 100
        
        return {
            "feature": feature_name,
            f"+{perturbation_percentage}% impact on Target (%)": mean_pct_change_up,
            f"-{perturbation_percentage}% impact on Target (%)": mean_pct_change_down
        }

    @staticmethod
    def Stepwise_Selection(df, target_col, method='forward', threshold=0.1):
        """
        Performs feature selection purely based on correlation metrics.
        Forward: Adds features with highest correlation to remaining residual.
        Backward: Eliminates features with lowest correlation to target/residual.
        """
        X_df = df.drop(columns=[target_col])
        y = df[target_col]
        
        features = list(X_df.columns)
        selected = []
        
        if method == 'forward':
            remaining = features.copy()
            current_residual = y.to_numpy()
            
            while remaining:
                # Find correlations of remaining features with current target residual
                corrs = [np.corrcoef(X_df[f].to_numpy(), current_residual)[0, 1] for f in remaining]
                abs_corrs = [abs(c) for c in corrs]
                max_idx = np.argmax(abs_corrs)
                
                if abs_corrs[max_idx] > threshold:
                    best_feat = remaining.pop(max_idx)
                    selected.append(best_feat)
                    
                    # Update residual by stripping out the effect of the newly selected feature
                    # via orthogonal projection (Gram-Schmidt concept)
                    f_vec = X_df[best_feat].to_numpy().reshape(-1, 1)
                    f_vec_norm = f_vec / np.linalg.norm(f_vec)
                    current_residual = (current_residual.reshape(-1, 1) - np.dot(current_residual, f_vec_norm) * f_vec_norm).flatten()
                else:
                    break
                    
        elif method == 'backward':
            selected = features.copy()
            while len(selected) > 1:
                # Check correlations of selected features with target
                corrs = [np.corrcoef(X_df[f].to_numpy(), y.to_numpy())[0, 1] for f in selected]
                abs_corrs = [abs(c) for c in corrs]
                min_idx = np.argmin(abs_corrs)
                
                if abs_corrs[min_idx] < threshold:
                    selected.pop(min_idx)
                else:
                    break
                    
        return selected

In [10]:
# 1. Generate Synthetic Dataset
np.random.seed(42)
num_samples = 200

# Features: X1 & X2 are highly predictive, X3 is weak noise, X4 is completely random
X1 = np.random.randn(num_samples)
X2 = np.random.randn(num_samples) + 0.5 * X1
X3 = np.random.randn(num_samples) * 0.1
X4 = np.random.randn(num_samples) * 10 

# Target Y depends purely on X1 and X2
Y = 3.5 + 2.0 * X1 - 1.5 * X2 + np.random.randn(num_samples) * 0.2

data = pd.DataFrame({'X1': X1, 'X2': X2, 'X3': X3, 'X4': X4, 'Target': Y})

# --- DIAGNOSTIC 1: Correlation Map ---
print("## Correlation Matrix Display")
print(data.corr().round(3))
print("-" * 50)

# --- EXPERIMENTATION: Correlation-based Forward Selection ---
print("## Executing Feature Selection")
chosen_features = AdvancedLinearRegression.Stepwise_Selection(data, target_col='Target', method='forward', threshold=0.15)
print(f"Selected Features by Engine: {chosen_features}")
print("-" * 50)

# --- MODEL TRAINING ---
X_selected = data[chosen_features]
y_target = data['Target']

model = AdvancedLinearRegression(fit_intercept=True)
model.fit(X_selected, y_target)

# --- DIAGNOSTIC 2: Return Coefficients ---
print("## Model Coefficients")
for feat, coef in model.get_coefficients().items():
    print(f"{feat:10}: {coef:.4f}")
print("-" * 50)

# --- DIAGNOSTIC 3: Sensitivity Simulation ---
print("## Sensitivity Simulation (What-if Analysis)")
# How sensitive is our target to a 10% change in X1?
sensitivity_x1 = model.simulate_sensitivity(X_selected, feature_name='X1', perturbation_percentage=10.0)
print(sensitivity_x1)
print("-" * 50)

# --- DIAGNOSTIC 4: Actual vs Predicted View ---
predictions = model.predict(X_selected)
comparison_df = pd.DataFrame({
    'Actual': y_target,
    'Predicted': predictions,
    'Residual': y_target - predictions
})
print("## Actual vs Predicted Sample Snapshot")
print(comparison_df.head(6).round(4))

## Correlation Matrix Display
           X1     X2     X3     X4  Target
X1      1.000  0.495 -0.134  0.065   0.558
X2      0.495  1.000 -0.084 -0.062  -0.439
X3     -0.134 -0.084  1.000  0.106  -0.056
X4      0.065 -0.062  0.106  1.000   0.142
Target  0.558 -0.439 -0.056  0.142   1.000
--------------------------------------------------
## Executing Feature Selection
Selected Features by Engine: ['X1', 'X2', 'X4']
--------------------------------------------------


TypeError: only 0-dimensional arrays can be converted to Python scalars